<a href="https://colab.research.google.com/github/zhuzhulala/zheng-wanzhu/blob/main/%E3%80%8CPrompt_ipynb%E3%80%8DLAN'S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI University Week 2：提示工程基礎與 Few-Shot 技巧

在工廠數位轉型與自動化系統（如 TMS、WCS 等）的開發中，我們常需要 AI 協助產出固定格式的代碼或架構建議。
本節課我們將學習如何透過 **System Prompt（系統提示詞）** 設定 AI 的人設，並比較 **Zero-shot（零樣本）** 與 **Few-shot（少樣本）** 的輸出差異。

In [6]:
# 安裝必要的套件並載入 API Key
!pip -q install openai

import os
from google.colab import userdata
from openai import OpenAI

# 取得 Groq API Key (請確保已在 Colab 左側的 Secrets 中設定 GROQ_API_KEY)
api_key = userdata.get('LANNER1ST')
if not api_key:
    raise ValueError("LANNER1ST")

os.environ["LANNER1ST"] = api_key

# 初始化 OpenAI 客戶端 (指向 Groq 的端點)
client = OpenAI(
    api_key=os.environ["LANNER1ST"],
    base_url="https://api.groq.com/openai/v1"
)

GROQ_MODEL = "llama-3.1-8b-instant"
#GROQ_MODEL = "llama-3.1-8b-instant"
print(f"環境設定完成，使用模型：{GROQ_MODEL}")


環境設定完成，使用模型：llama-3.1-8b-instant


## 📝 1. 設計 System Prompt 與 User Prompt
我們將 AI 設定為「智慧製造與工廠自動化顧問」，並要求它以特定格式回答。

In [ ]:
# 1) System Prompt：規範 AI 的身份、口吻、格式
system_prompt = """
你是一位專精於『智慧製造與工廠自動化』的資深AI助教。
請用繁體中文回答，語氣活潑但具備工程專業。
回答格式必須嚴格遵循以下3點：
1. 問題理解：簡述使用者的痛點。
2. 建議做法（3點）：給出具體的系統設計或架構建議。
3. 可執行的範例：給出一段 Python 或 JSON 格式的實作小範例。
回答盡量精簡，總字數控制在 250 字以內。
""".strip()

# 2) User Prompt：使用者實際的提問
user_prompt = """
我們部門準備導入新的專案管理與設備異常通報機制，我要在中控室做一個異常通報看板，請給我一個 AI 可以協助的設計方向。
""".strip()

# 3) Few-Shot 範例：為了讓 AI 徹底明白我們想要的「可執行範例」長怎樣，我們給它一個對話示範
example_user = "我要做一個機台溫度監控的通報模組，請給建議。"
example_assistant = """
1. 問題理解
你想建立一個主動式的溫度監控機制，避免機台過熱導致停機。

2. 建議做法（3點）
- 制定閾值邊界（如：黃燈警示、紅燈停機）。
- 整合邊緣運算（Edge Computing）先過濾雜訊，再上傳異常數據。
- 發生異常時，觸發 Webhook 推播至 LINE Bot 或中控螢幕。

3. 可執行的範例
以下是推播至 LINE Notify 的 JSON 格式範例：
```json
{
  "machine_id": "AGV-001",
  "error_code": "E-TEMP-099",
  "level": "Critical",
  "message": "驅動馬達溫度過高 (85°C)，請立即派工檢查！"
}
```
""".strip()


## 🚀 2. 構建對話歷史 (Messages) 並呼叫 API
在呼叫 API 時，有沒有把 `example_user` 和 `example_assistant` 放進去，就是 **Zero-Shot** 和 **Few-Shot** 的最大差別。

In [11]:
def chat_with_groq(messages, model=GROQ_MODEL, temperature=0.3, max_tokens=512):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content

# 1) System Prompt：規範 AI 的身份、口吻、格式
system_prompt = """
你是一位專精於『智慧製造與工廠自動化』的資深AI助教。
請用繁體中文回答，語氣活潑但具備工程專業。
回答格式必須嚴格遵循以下3點：
1. 問題理解：簡述使用者的痛點。
2. 建議做法（3點）：給出具體的系統設計或架構建議。
3. 可執行的範例：給出一段 Python 或 JSON 格式的實作小範例。
回答盡量精簡，總字數控制在 250 字以內。
""".strip()

# 2) User Prompt：使用者實際的提問
user_prompt = """
我們部門準備導入新的專案管理與設備異常通報機制，我要在中控室做一個異常通報看板，請給我一個 AI 可以協助的設計方向。
""".strip()

# 3) Few-Shot 範例：為了讓 AI 徹底明白我們想要的「可執行範例」長怎樣，我們給它一個對話示範
example_user = "我要做一個機台溫度監控的通報模組，請給建議。"
example_assistant = """
1. 問題理解
你想建立一個主動式的溫度監控機制，避免機台過熱導致停機。

2. 建議做法（3點）
- 制定閾值邊界（如：黃燈警示、紅燈停機）。
- 整合邊緣運算（Edge Computing）先過濾雜訊，再上傳異常數據。
- 發生異常時，觸發 Webhook 推播至 LINE Bot 或中控螢幕。

3. 可執行的範例
以下是推播至 LINE Notify 的 JSON 格式範例：
```json
{
  "machine_id": "AGV-001",
  "error_code": "E-TEMP-099",
  "level": "Critical",
  "message": "驅動馬達溫度過高 (85°C)，請立即派工檢查！"
}
```
""".strip()

# 【測試 A】Zero-Shot (完全不給範例)
messages_zero_shot = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# 【測試 B】Few-Shot (塞入一組完美的問答範例)
messages_few_shot = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example_user},
    {"role": "assistant", "content": example_assistant},
    {"role": "user", "content": user_prompt}
]

tests = [
    ("A. Zero-Shot (無對話範例)", messages_zero_shot),
    ("B. Few-Shot (有對話範例)", messages_few_shot),
]

results = {}

for title, msgs in tests:
    print("=" * 60)
    print(title)
    print(msgs)
    print("=" * 60)
    try:
        output = chat_with_groq(msgs)
        results[title] = output
        print(output)
    except Exception as e:
        results[title] = f"ERROR: {e}"
        print("發生錯誤：", e)
    print("\n")

A. Zero-Shot (無對話範例)
[{'role': 'system', 'content': '你是一位專精於『智慧製造與工廠自動化』的資深AI助教。\n請用繁體中文回答，語氣活潑但具備工程專業。\n回答格式必須嚴格遵循以下3點：\n1. 問題理解：簡述使用者的痛點。\n2. 建議做法（3點）：給出具體的系統設計或架構建議。\n3. 可執行的範例：給出一段 Python 或 JSON 格式的實作小範例。\n回答盡量精簡，總字數控制在 250 字以內。'}, {'role': 'user', 'content': '我們部門準備導入新的專案管理與設備異常通報機制，我要在中控室做一個異常通報看板，請給我一個 AI 可以協助的設計方向。'}]
**問題理解**

你需要在中控室設計一個異常通報看板，讓人員能夠快速地檢視設備異常通報的資訊，提高工作效率和應急反應速度。

**建議做法**

1. **資料整合**: 建立一個資料平台，整合來自各個設備的異常通報資訊，包括設備名稱、異常類型、發生時間等。
2. **視覺化呈現**: 使用可視化工具（如 Dashboard 或 KPI 表格），呈現異常通報的資訊，讓人員能夠快速地檢視和識別異常情況。
3. **AI 驅動的預警**: 利用機器學習和預測模型，預測設備可能發生的異常，提前發出預警，幫助人員提前處理和解決問題。

**可執行的範例**

```python
import pandas as pd
import matplotlib.pyplot as plt

# 假設資料表格
data = {
    '設備名稱': ['機器A', '機器B', '機器C'],
    '異常類型': ['溫度過高', '壓力過低', '故障'],
    '發生時間': ['2023-03-01 08:00', '2023-03-01 09:00', '2023-03-01 10:00']
}

df = pd.DataFrame(data)

# 顯示資料表格
print(df)

# 顯示視覺化圖表
plt.bar(df['設備名稱'], df['發生時間'])
plt.xlabel('設備名稱')
plt.ylabel('發生時間')
plt.title('設備異常通報')
plt.show()
```

這個範

## 📊 3. 輸出品質驗證
我們寫一個簡單的 Python 腳本，來檢查 AI 的回答有沒有「聽話」遵循我們規定的格式。通常 Few-Shot 的表現會穩定非常多！

In [ ]:
def quick_check(text):
    # 檢查是否有遵循三大結構
    has_understanding = "問題理解" in text
    has_suggestions = "建議做法" in text
    has_example = "範例" in text or "```" in text
    is_short = len(text) < 500

    return {
        "格式_問題理解": "✅" if has_understanding else "❌",
        "格式_建議做法": "✅" if has_suggestions else "❌",
        "格式_實作範例": "✅" if has_example else "❌",
        "字數是否達標": "✅" if is_short else "❌"
    }

for k, v in results.items():
    print(f"【{k}】評估結果：")
    if not v.startswith("ERROR:"):
        eval_result = quick_check(v)
        for criterion, passed in eval_result.items():
            print(f"  {criterion}: {passed}")
        print(f"  總字數：{len(v)} 字\n")
    else:
        print("API 呼叫失敗，無法評估。\n")
